# GeoReasoner

GeoReasoner:
Grounded Multimodal Reasoning for Business Profile Freshness Verification

## Project Goal

GeoReasoner is a multimodal reasoning system that combines retrieval, grounding, visual evidence generation, verification, and agentic reasoning to determine whether newly arriving visual evidence supports, contradicts, or is insufficient to validate existing business profile metadata.

### Agentic Reasoning

When verification returns INSUFFICIENT_EVIDENCE, the system gathers additional evidence and re-evaluates the business profile before reaching a final decision.

The system evolves through several phases:

0. Foundation Models
1. Retrieval
2. Grounding
3. Evidence Generation
4. Structured Reasoning
5. Evidence Aggregation
6. Confidence Calibration
7. Verification
8. Evaluation
9. Agentic Freshness Verification

### End-to-End Workflow

User-Generated Images + Business Metadata + User Query
        →
Retrieval
        →
Grounding
        →
Evidence Generation
        →
Structured Reasoning
        →
Evidence Aggregation
        →
Confidence Calibration
        →
Verification
        →
Final Assessment

## System Architecture

        Current Business Profile Metadata
                       +
              New Visual Evidence

                       │
                       ▼

          ┌─────────────────────────────┐
          │ Retrieval Layer             │
          │ Evidence Retrieval          │
          └─────────────────────────────┘
                       │
                       ▼

          ┌─────────────────────────────┐
          │ Grounding Layer             │
          │ Metadata Validation         │
          └─────────────────────────────┘
                       │
                       ▼

          ┌─────────────────────────────┐
          │ Evidence Generation Layer   │
          │ BLIP Captioning             │
          └─────────────────────────────┘
                       │
                       ▼

          ┌─────────────────────────────┐
          │ Structured Reasoning Layer  │
          │ Explanation Generation      │
          └─────────────────────────────┘
                       │
                       ▼

          ┌─────────────────────────────┐
          │ Evidence Aggregation Layer  │
          │ Multi-Signal Fusion         │
          └─────────────────────────────┘
                       │
                       ▼

          ┌─────────────────────────────┐
          │ Confidence Calibration      │
          │ Grounding-Aware Confidence  │
          └─────────────────────────────┘
                       │
                       ▼

          ┌─────────────────────────────┐
          │ Verification Layer          │
          │ Freshness Verification      │
          └─────────────────────────────┘
                       │
                       ▼

          ┌─────────────────────────────┐
          │ Final Assessment            │
          │ SUPPORTED                   │
          │ CONTRADICTED                │
          │ INSUFFICIENT_EVIDENCE       │
          └─────────────────────────────┘

| Phase | Purpose | Input | Output |
|---------|---------|---------|---------|
| Foundation Models | Create a shared vision-language representation space | User-Generated Images + Business Categories | Image Embeddings, Text Embeddings |
| Retrieval | Retrieve evidence relevant to a user query | User Query | Top-K Candidate Images |
| Grounding | Validate whether visual evidence supports metadata | Image + Business Metadata | Consistency Score |
| Evidence Generation | Extract interpretable visual evidence | User-Generated Images | Natural Language Caption |
| Structured Reasoning | Explain decisions using evidence and metadata | Caption + Metadata + Consistency Score | Human-Readable Explanation |
| Evidence Aggregation | Combine multiple evidence sources into a unified assessment | Caption + Metadata + Consistency Score | Assessment + Preliminary Confidence |
| Confidence Calibration | Adjust confidence based on evidence quality and grounding strength | Assessment + Evidence Signals | Calibrated Confidence Estimate |
| Verification | Determine whether evidence supports, contradicts, or is insufficient for a claim | Caption + Metadata + Consistency Score | Support / Contradict / Insufficient Evidence |
| Evaluation | Verification Outputs + Human Assessment | Qualitative Analysis and Failure Modes |
| Agentic Freshness Verification | Gather additional evidence when verification is inconclusive | Business Profile + Candidate Images | Final Freshness Decision |




## Project Evolution

Retrieval
→ Grounding
→ Structured Reasoning
→ Verification
→ Evidence-First Verification
→ Agentic Freshness Verification

# Phase 0 — Foundation Models

## Objective

Establish the multimodal representation layer used by all subsequent phases.

## Components

CLIP
- Image encoder
- Text encoder

BLIP
- Caption generator

## Why This Exists

All later stages depend on shared image-text representations.

### Install Libraries

In [ ]:
!pip install transformers sentence-transformers matplotlib pillow opendatasets kaggle

In [ ]:
import os

# Dataset access configuration
# Credentials should be stored in Colab Secrets or environment variables.
os.environ["KAGGLE_API_TOKEN"] = "KGAT_79dc2387a2aa07c016739ca6fbda63a9" # Set KAGGLE_API_TOKEN in Colab Secrets
!kaggle datasets list -s mapillary

In [ ]:
import os

os.makedirs("data/images", exist_ok=True)

In [ ]:
import requests

image_urls = [
    "https://images.unsplash.com/photo-1555396273-367ea4eb4db5",
    "https://images.unsplash.com/photo-1528698827591-e19ccd7bc23d",
    "https://images.unsplash.com/photo-1517248135467-4c7edcad34c4",
    "https://images.unsplash.com/photo-1495474472287-4d71bcdd2085",
    "https://images.unsplash.com/photo-1504674900247-0877df9cc836"
]

for idx, url in enumerate(image_urls):
    response = requests.get(url)

    with open(f"data/images/img_{idx}.jpg", "wb") as f:
        f.write(response.content)

print("done")

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import os

image_files = os.listdir("data/images")

fig, axes = plt.subplots(1, len(image_files), figsize=(20,5))

for ax, img_file in zip(axes, image_files):
    img = Image.open(f"data/images/{img_file}")
    ax.imshow(img)
    ax.axis("off")

plt.show()

In [ ]:
metadata = {
    "img_0.jpg": "restaurant",
    "img_1.jpg": "coffee shop",
    "img_2.jpg": "restaurant",
    "img_3.jpg": "bakery",
    "img_4.jpg": "grocery store"
}

In [ ]:
!pip install transformers sentence-transformers torch

In [ ]:
import torch
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import os

In [ ]:
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

### Load one image

In [ ]:
image_path = "data/images/img_0.jpg"

image = Image.open(image_path)

image

### Generate Image Embedding



In [ ]:
inputs = processor(images=image, return_tensors="pt")

with torch.no_grad():
    image_outputs = model.get_image_features(**inputs)

image_features = image_outputs.pooler_output

print(image_features.shape)

### Generate Text Embedding

In [ ]:
categories = [
    "coffee shop",
    "restaurant",
    "pharmacy",
    "bakery",
    "gas station",
    "grocery store",
    "hotel",
    "bookstore"
]

text_inputs = processor(
    text=categories,
    return_tensors="pt",
    padding=True
)

with torch.no_grad():
    text_outputs = model.get_text_features(**text_inputs)

text_features = text_outputs.pooler_output

print(text_features.shape)

In [ ]:
image_features = image_features / image_features.norm(dim=-1, keepdim=True)
text_features = text_features / text_features.norm(dim=-1, keepdim=True)

In [ ]:
similarities = torch.matmul(image_features, text_features.T)

for category, score in zip(categories, similarities[0]):
    print(category, float(score))

# Phase 1 — Multimodal Retrieval

## Question

Can we retrieve businesses relevant to a user query?

## Problem Statement

Business profiles become stale over time.

Businesses may close, change categories, remodel, update branding, or modify their offerings.

Meanwhile, new visual evidence continuously arrives from sources such as User-Generated Images and street-level imagery.

GeoReasoner evaluates whether newly arriving visual evidence remains consistent with existing business metadata and helps determine whether a business profile should be retained, reviewed, or updated.

## Input

Query:
"coffee shop"

## Output

Top-k relevant User-Generated Images

## Architecture

* Query --> Text Embedding
* Image Database --> Image Embeddings
* Similarity Search --> Top K Results

## Generate Image Embeddings

In [ ]:
image_embeddings = []
image_paths = []

for img_file in os.listdir("data/images"):

    path = f"data/images/{img_file}"

    image = Image.open(path)

    inputs = processor(images=image, return_tensors="pt")

    with torch.no_grad():
        outputs = model.get_image_features(**inputs)

    features = outputs.pooler_output

    features = features / features.norm(dim=-1, keepdim=True)

    image_embeddings.append(features)
    image_paths.append(path)

## Build Retrieval Index



In [ ]:
image_embeddings = torch.cat(image_embeddings, dim=0)

print(image_embeddings.shape)

## Implement Retrieval Engine

In [ ]:
def retrieve_images(query, top_k=3):

    text_inputs = processor(
        text=[query],
        return_tensors="pt",
        padding=True
    )

    with torch.no_grad():
        text_outputs = model.get_text_features(**text_inputs)

    text_features = text_outputs.pooler_output

    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

    similarities = torch.matmul(
        image_embeddings,
        text_features.T
    ).squeeze()

    top_indices = similarities.topk(top_k).indices

    return [(image_paths[i], similarities[i].item()) for i in top_indices]

## Retrieval Example


In [ ]:
results = retrieve_images("coffee shop")

results

### Example

Input Query:
coffee shop

Output:
Top matching User-Generated Images ranked by semantic similarity.

This demonstrates retrieval over a shared vision-language embedding space.

## Visualize Results



In [ ]:
fig, axes = plt.subplots(1, len(results), figsize=(15,5))

for ax, (path, score) in zip(axes, results):

    img = Image.open(path)

    ax.imshow(img)
    ax.set_title(f"{score:.2f}")
    ax.axis("off")

plt.show()

# Phase 2 — Metadata Grounding

## Question

Does visual evidence support the business metadata?

## Motivation

Retrieval alone is insufficient.

A retrieved image may not actually support
the claimed business category.

We need a grounding mechanism.

### Example

* Input: Image: Coffee Shop
* Metadata: Coffee Shop
* Output: Consistency Score: 0.253

## Architecture

* Image --> Embedding
* Metadata --> Embedding
* Similarity --> Consistency Score

## Metadata Definition

In [ ]:
metadata = {
    "img_0.jpg": "restaurant",
    "img_1.jpg": "coffee shop",
    "img_2.jpg": "restaurant",
    "img_3.jpg": "bakery",
    "img_4.jpg": "grocery store"
}

## Consistency Scoring



In [ ]:
def check_consistency(image_path, metadata_label):

    image = Image.open(image_path)

    image_inputs = processor(
        images=image,
        return_tensors="pt"
    )

    with torch.no_grad():
        image_outputs = model.get_image_features(**image_inputs)

    image_features = image_outputs.pooler_output

    image_features = (
        image_features /
        image_features.norm(dim=-1, keepdim=True)
    )

    text_inputs = processor(
        text=[metadata_label],
        return_tensors="pt",
        padding=True
    )

    with torch.no_grad():
        text_outputs = model.get_text_features(**text_inputs)

    text_features = text_outputs.pooler_output

    text_features = (
        text_features /
        text_features.norm(dim=-1, keepdim=True)
    )

    similarity = torch.matmul(
        image_features,
        text_features.T
    ).item()

    return similarity

## Grounding Evaluation — Positive Examples

In [ ]:
for img_name, label in metadata.items():

    path = f"data/images/{img_name}"

    score = check_consistency(path, label)

    print(img_name, label, score)

## Grounding Evaluation — Negative Examples



In [ ]:
bad_examples = [
    ("img_0.jpg", "pharmacy"),
    ("img_1.jpg", "gas station"),
    ("img_2.jpg", "bookstore")
]

for img_name, label in bad_examples:

    path = f"data/images/{img_name}"

    score = check_consistency(path, label)

    print(img_name, label, score)

### Observation

Matching metadata generally produces higher similarity scores than mismatched metadata.

This provides a quantitative grounding signal that can be used for reasoning and confidence estimation.

# Phase 3 — Visual Evidence Generation

## Question

What visual evidence can we extract from the image?

## Motivation

Similarity scores are not interpretable.

Humans need understandable evidence.

## Inputs

User-Generated Images

## Outputs

Natural Language Caption

### Example

* Input: User-Generated Images
* Output: "a bicycle parked in front of a store"

## Architecture

Image --> BLIP --> Caption

## Load Visual Captioning Model

In [ ]:
!pip install transformers accelerate

In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration

In [ ]:
blip_processor = BlipProcessor.from_pretrained(
    "Salesforce/blip-image-captioning-base"
)

blip_model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base"
)

## Generate Visual Evidence

In [ ]:
inputs = blip_processor(
    images=image,
    return_tensors="pt"
)

outputs = blip_model.generate(**inputs)

caption = blip_processor.decode(
    outputs[0],
    skip_special_tokens=True
)

print(caption)

### Example

Generated Caption:

"a bicycle parked in front of a store"

This caption becomes visual evidence used during reasoning.

## Initial Explanation Generation

In [ ]:
def generate_reasoning(image_path, metadata_label):

    image = Image.open(image_path)

    inputs = blip_processor(
        images=image,
        return_tensors="pt"
    )

    outputs = blip_model.generate(**inputs)

    caption = blip_processor.decode(
        outputs[0],
        skip_special_tokens=True
    )

    score = check_consistency(
        image_path,
        metadata_label
    )

    explanation = f"""
Metadata label: {metadata_label}

Generated visual description: {caption}

Consistency score: {score:.3f}
"""

    return explanation

## Structured Reasoning Engine

In [ ]:
print(generate_reasoning(
    "data/images/img_1.jpg",
    "coffee shop"
))

# Phase 4 — Structured Reasoning
## Question

Can the system explain its decision?

## Motivation

Grounding scores alone are insufficient.

The system should provide
human-readable reasoning.

## Inputs

Caption
Metadata
Consistency Score

## Outputs

* Business Profile Assessment
* Human-Readable Explanation

### Example

* Input - Caption: a bicycle parked in front of a store
* Metadata: coffee shop
* Consistency: 0.253
* Output: Highly Consistent

## Architecture

Caption + Metadata + Consistency Score --> Reasoning

## Why Structured Reasoning?

Similarity scores alone are difficult to interpret.

The system should generate human-readable assessments that explain whether visual evidence supports the metadata.

This transforms the system from a retrieval engine into a reasoning system.

## Structured Reasoning Engine

In [ ]:
def generate_structured_reasoning(image_path, metadata_label):

    image = Image.open(image_path)

    # Generate caption
    inputs = blip_processor(
        images=image,
        return_tensors="pt"
    )

    outputs = blip_model.generate(**inputs)

    caption = blip_processor.decode(
        outputs[0],
        skip_special_tokens=True
    )

    # Consistency score
    score = check_consistency(
        image_path,
        metadata_label
    )

    # Rule-based reasoning
    if score > 0.24:
        assessment = "Highly consistent"
    elif score > 0.20:
        assessment = "Moderately consistent"
    else:
        assessment = "Weakly consistent"

    reasoning = f"""
Business category: {metadata_label}

Visual evidence:
- {caption}

Consistency assessment:
{assessment}

Consistency score:
{score:.3f}
"""

    return reasoning

## Structured Reasoning Example

In [ ]:
print(
    generate_structured_reasoning(
        "data/images/img_1.jpg",
        "coffee shop"
    )
)

### Example

Input

Caption:
a bicycle parked in front of a store

Metadata:
coffee shop

Consistency Score:
0.253

Output

Business category:
coffee shop

Visual evidence:
a bicycle parked in front of a store

Consistency assessment:
Highly consistent

## Reasoning Under Contradictory Metadata

In [ ]:
print(
    generate_structured_reasoning(
        "data/images/img_1.jpg",
        "gas station"
    )
)

### Observation

The same visual evidence may produce different reasoning outcomes depending on the metadata claim.

This motivates confidence estimation and verification.

# Phase 5 — Evidence Aggregation

## Question

Can we combine multiple evidence sources?

## Motivation

Reasoning should not depend on
a single signal.

## Inputs

Caption
Metadata
Consistency Score

## Output

Assessment
Confidence

## Architecture

Caption + Metadata + Consistency --> Evidence Aggregation --> Confidence

## Why Evidence Aggregation?
No single signal is sufficient.

The system combines:

- Visual Evidence
- Metadata
- Consistency Scores

to generate more robust assessments.

This introduces evidence aggregation.

## Hybrid Reasoning Engine

In [ ]:
def generate_reasoning(image_path, metadata_label):

    image = Image.open(image_path)

    # BLIP caption
    inputs = blip_processor(
        images=image,
        return_tensors="pt"
    )

    outputs = blip_model.generate(**inputs)

    caption = blip_processor.decode(
        outputs[0],
        skip_special_tokens=True
    )

    # Consistency score
    score = check_consistency(
        image_path,
        metadata_label
    )

    # Confidence logic
    if score > 0.24:
        confidence = "High"
        assessment = (
            "The image appears strongly consistent "
            "with the provided business category."
        )

    elif score > 0.20:
        confidence = "Moderate"
        assessment = (
            "The image appears partially consistent "
            "with the provided business category, "
            "though explicit visual evidence is limited."
        )

    else:
        confidence = "Low"
        assessment = (
            "The image appears weakly related "
            "to the provided business category."
        )

    reasoning = f"""
Business category:
{metadata_label}

Visual evidence:
- {caption}

Semantic consistency score:
{score:.3f}

Assessment:
{assessment}

Confidence:
{confidence}
"""

    return reasoning

## Hybrid Reasoning Example

In [ ]:
print(
    generate_reasoning(
        "data/images/img_1.jpg",
        "coffee shop"
    )
)

# Phase 6 — Confidence Calibration

## Question

How confident should the system be?

## Motivation

Similarity scores may be high
even when visual evidence is weak.

Confidence should reflect
evidence quality.

## Example

Caption:
"a bicycle parked in front of a store"

Metadata:
"coffee shop"

Similarity:
0.253

Result:
Moderate Confidence

## Architecture

Similarity + Caption Evidence + Metadata --> Confidence

## Why Confidence Calibration?

Similarity scores alone can be misleading.

The system should evaluate whether visual evidence actually supports the business category.

Weak evidence should reduce confidence even when similarity is relatively high.

This introduces grounding-aware confidence estimation.

## Evidence Keywords

In [ ]:
category_keywords = {
    "coffee shop": [
        "coffee",
        "cafe",
        "cup",
        "espresso",
        "pastry"
    ],

    "restaurant": [
        "restaurant",
        "table",
        "food",
        "dining"
    ],

    "bakery": [
        "bread",
        "bakery",
        "pastry",
        "cake"
    ],

    "grocery store": [
        "grocery",
        "market",
        "store",
        "produce"
    ]
}

### Example

Caption:
a bicycle parked in front of a store

Expected Evidence for Coffee Shop:

- coffee
- cafe
- cup
- espresso
- pastry

Result:

No supporting keywords detected.

Confidence should decrease.

This prevents overconfident reasoning.

# Phase 7 — Verification

# Phase 7A — Verification Layer

## Question

Does the available evidence support the metadata claim?

## Motivation

Reasoning systems should verify claims before generating conclusions.

Verification provides an additional safeguard against unsupported or contradictory outputs.

## Inputs

Caption

Metadata

Consistency Score

## Outputs

SUPPORTED

CONTRADICTED

INSUFFICIENT_EVIDENCE

## Architecture

Caption + Metadata + Consistency Score
                ↓
          Verification
                ↓
         Verification Label

In [ ]:
def verify_claim(
    caption,
    metadata_label,
    consistency_score
):

    keywords = category_keywords.get(
        metadata_label,
        []
    )

    keyword_matches = sum(
        keyword in caption.lower()
        for keyword in keywords
    )

    if keyword_matches >= 2 and consistency_score > 0.24:
        return "SUPPORTED"

    elif keyword_matches == 0 and consistency_score < 0.22:
        return "CONTRADICTED"

    else:
        return "INSUFFICIENT_EVIDENCE"

In [ ]:
caption = "a bicycle parked in front of a store"

score = 0.253

verification = verify_claim(
    caption,
    "coffee shop",
    score
)

print(verification)

In [ ]:
def generate_verified_reasoning(
    image_path,
    metadata_label
):

    image = Image.open(image_path)

    inputs = blip_processor(
        images=image,
        return_tensors="pt"
    )

    outputs = blip_model.generate(**inputs)

    caption = blip_processor.decode(
        outputs[0],
        skip_special_tokens=True
    )

    score = check_consistency(
        image_path,
        metadata_label
    )

    verification = verify_claim(
        caption,
        metadata_label,
        score
    )

    reasoning = f"""
Business category:
{metadata_label}

Visual evidence:
{caption}

Consistency score:
{score:.3f}

Verification:
{verification}
"""

    return reasoning

In [ ]:
print(
    generate_verified_reasoning(
        "data/images/img_1.jpg",
        "coffee shop"
    )
)

### Example

Claim

coffee shop

Visual Evidence

"a bicycle parked in front of a store"

Consistency Score

0.253

Verification Result

INSUFFICIENT_EVIDENCE

Observation

Although the image embedding is moderately aligned with the coffee shop category, the generated visual evidence does not contain explicit coffee-shop indicators.

The system therefore declines to support the claim and returns INSUFFICIENT_EVIDENCE.

This demonstrates the value of verification beyond similarity scoring alone.

In [ ]:
def verify_claim_explainable(
    caption,
    metadata_label,
    consistency_score
):

    keywords = category_keywords.get(
        metadata_label,
        []
    )

    matched_keywords = [
        keyword
        for keyword in keywords
        if keyword in caption.lower()
    ]

    if len(matched_keywords) >= 2 and consistency_score > 0.24:

        return {
            "verification": "SUPPORTED",
            "reason":
                f"Found evidence keywords: {matched_keywords}"
        }

    elif len(matched_keywords) == 0 and consistency_score < 0.22:

        return {
            "verification": "CONTRADICTED",
            "reason":
                "No supporting keywords detected."
        }

    else:

        return {
            "verification": "INSUFFICIENT_EVIDENCE",
            "reason":
                "Visual evidence is too weak to support the claim."
        }

In [ ]:
def generate_verified_reasoning_v2(
    image_path,
    metadata_label
):

    image = Image.open(image_path)

    inputs = blip_processor(
        images=image,
        return_tensors="pt"
    )

    outputs = blip_model.generate(**inputs)

    caption = blip_processor.decode(
        outputs[0],
        skip_special_tokens=True
    )

    score = check_consistency(
        image_path,
        metadata_label
    )

    verification_result = verify_claim_explainable(
        caption,
        metadata_label,
        score
    )

    reasoning = f"""
Business category:
{metadata_label}

Visual evidence:
{caption}

Consistency score:
{score:.3f}

Verification:
{verification_result["verification"]}

Verification reason:
{verification_result["reason"]}
"""

    return reasoning

In [ ]:
print(
    generate_verified_reasoning_v2(
        "data/images/img_1.jpg",
        "coffee shop"
    )
)

### Example

Claim

coffee shop

Visual Evidence

"a bicycle parked in front of a store"

Consistency Score

0.253

Verification

INSUFFICIENT_EVIDENCE

Reason

Visual evidence is too weak to support the claim.

Observation

The retrieval and grounding layers suggest that the image may be related to a coffee shop. However, the generated visual evidence does not contain explicit coffee-shop indicators such as coffee, cafe, espresso, cup, or pastry.

The verification layer therefore declines to support the claim and returns INSUFFICIENT_EVIDENCE.

This demonstrates how verification can prevent unsupported conclusions even when similarity scores are relatively high.

# Phase 7B — LLM-Based Verification

## Question

Can a language model verify claims using visual evidence and grounding information?

## Motivation

Rule-based verification is simple but limited.

A language model can reason about evidence, metadata, and grounding signals together.

## Inputs

Caption

Metadata

Consistency Score

## Outputs

SUPPORTED

CONTRADICTED

INSUFFICIENT_EVIDENCE

## Architecture

Caption + Metadata + Consistency Score --> LLM Verifier --> Verification + Explanation

In [ ]:
!pip install transformers accelerate

In [ ]:
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

llm = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map={"": 0}
)

print(next(llm.parameters()).device) # check all the parameteres are on GPU


In [ ]:
!nvidia-smi # check memory usage after loading the model

In [ ]:
prompt = "Say hello in one sentence."

inputs = tokenizer(
    prompt,
    return_tensors="pt"
)

inputs = {
    k: v.to("cuda")
    for k, v in inputs.items()
}

with torch.no_grad():
    outputs = llm.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=False
    )

print(
    tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )
)

In [ ]:
def build_verification_prompt(
    caption,
    metadata_label,
    consistency_score
):

    prompt = f"""
You are a multimodal verification system.

Business category:
{metadata_label}

Visual evidence:
{caption}

Consistency score:
{consistency_score:.3f}

Determine whether the evidence:

SUPPORTED
CONTRADICTED
INSUFFICIENT_EVIDENCE

Return your answer in the format:

Verification: <label>

Reason: <short explanation>
"""

    return prompt

In [ ]:
prompt = build_verification_prompt(
    caption="a bicycle parked in front of a store",
    metadata_label="coffee shop",
    consistency_score=0.253
)

print(prompt)

In [ ]:
def llm_verify(
    caption,
    metadata_label,
    consistency_score,
    max_new_tokens=80
):

    prompt = f"""
You are a verification system.

Business category:
{metadata_label}

Visual evidence:
{caption}

Consistency score:
{consistency_score:.3f}

Task:

Determine exactly one of:

SUPPORTED
CONTRADICTED
INSUFFICIENT_EVIDENCE

Respond using ONLY:

Verification: <label>

Reason: <one sentence>
"""

    messages = [
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    )

    inputs = {
        k: v.to("cuda")
        for k, v in inputs.items()
    }

    with torch.no_grad():
        outputs = llm.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated,
        skip_special_tokens=True
    )

In [ ]:
result = llm_verify(
    caption="a bicycle parked in front of a store",
    metadata_label="coffee shop",
    consistency_score=0.253
)

print(result)

### Observation: Claim-Conditioned Hallucination

The LLM verifier incorrectly returned **SUPPORTED** even though the visual evidence did not contain any coffee-shop-specific indicators.

Input:

- Metadata: coffee shop
- Caption: "a bicycle parked in front of a store"
- Consistency Score: 0.253

LLM Output:

- Verification: SUPPORTED
- Reason: The model inferred the presence of a coffee shop despite the absence of supporting evidence in the caption.

Analysis:

The verifier was exposed to the metadata claim before performing evidence analysis. As a result, the model treated the claim as likely true and generated a justification that was not grounded in the observed evidence.

This behavior illustrates a common failure mode in LLM-based reasoning systems known as **claim-conditioned hallucination**, where the model produces explanations that support a claim rather than objectively evaluating the available evidence.

### Design Improvement

To mitigate this issue, GeoReasoner adopts an Evidence-First Verification architecture:

Caption
→ Evidence Extraction
→ Structured Evidence
→ Claim Verification
→ SUPPORTED / CONTRADICTED / INSUFFICIENT_EVIDENCE

By separating evidence extraction from claim verification, the system encourages reasoning based on explicitly observed information rather than assumptions derived from metadata.

### Key Takeaway

The experiment demonstrates that adding an LLM verifier alone does not guarantee grounded reasoning. Robust verification requires architectural safeguards that force the model to reason from evidence before evaluating claims.

# Phase 7C — Evidence-First Verification

## Question

Can verification be improved by separating evidence extraction from claim evaluation?

## Motivation

Direct LLM verification is susceptible to claim-conditioned hallucination.

To improve robustness, GeoReasoner first extracts evidence from the caption and then verifies the metadata claim using only the extracted evidence.

## Inputs

Caption

Metadata

Consistency Score

## Outputs

SUPPORTED

CONTRADICTED

INSUFFICIENT_EVIDENCE

## Architecture

Caption
→ Evidence Extraction
→ Structured Evidence
→ Claim Verification
→ SUPPORTED / CONTRADICTED / INSUFFICIENT_EVIDENCE

In [ ]:
def extract_evidence(caption):

    prompt = f"""
You are an evidence extraction system.

Caption:
{caption}

Rules:

- Extract only facts explicitly present.
- Do not infer business type.
- Do not infer objects not mentioned.
- Do not add background knowledge.

Return:

Objects:
- ...

Attributes:
- ...

Actions:
- ...
"""

    messages = [
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    )

    inputs = {
        k: v.to("cuda")
        for k, v in inputs.items()
    }

    with torch.no_grad():
        outputs = llm.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated,
        skip_special_tokens=True)

In [ ]:
evidence = extract_evidence(
    "a bicycle parked in front of a store"
)

print(evidence)

### Observation

The evidence extraction stage successfully identified only information explicitly present in the caption.

Extracted Evidence

- Bicycle
- Front of store
- Parked

Notably, the extractor did not infer a business category or introduce unsupported concepts such as coffee shop, restaurant, or retail store.

This behavior is desirable because it separates evidence collection from claim evaluation and reduces the risk of claim-conditioned hallucination in downstream verification.

In [ ]:
def verify_from_evidence(
    evidence,
    metadata_label,
    consistency_score
):

    prompt = f"""
You are a strict verification system.

Business category claim:
{metadata_label}

Extracted evidence:
{evidence}

Consistency score:
{consistency_score:.3f}

Task:

Step 1:
List all pieces of evidence that directly support the claim.

Step 2:
Count how many supporting pieces of evidence exist.

Step 3:
Determine:

SUPPORTED
CONTRADICTED
INSUFFICIENT_EVIDENCE

Rules:

- Use ONLY the extracted evidence.
- Never assume facts not present.
- Never rename objects.
- Never infer business type.
- If no supporting evidence exists, return INSUFFICIENT_EVIDENCE.

Return exactly:

Supporting Evidence:
- ...

Verification:
...

Reason:
...
"""

    messages = [
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    )

    inputs = {
        k: v.to("cuda")
        for k, v in inputs.items()
    }

    with torch.no_grad():
        outputs = llm.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated,
        skip_special_tokens=True)

In [ ]:
result = verify_from_evidence(
    evidence=evidence,
    metadata_label="coffee shop",
    consistency_score=0.253
)

print(result)

## Conclusion
LLM-based verification remained susceptible to claim-conditioned hallucination even after introducing an evidence-extraction stage.

Future work includes constrained verification, evidence citation, and hybrid rule-based guardrails to prevent unsupported evidence attribution.

### Future Work

The evidence-first verifier reduced direct exposure to raw captions but did not fully eliminate claim-conditioned hallucination.

Experiments showed that the verifier continued to incorporate metadata assumptions into its reasoning, even when the extracted evidence did not explicitly support the claim.

Potential improvements include:

- Evidence citation and attribution
- Constrained verification using structured evidence
- Hybrid rule-based and LLM verification
- Chain-of-verification prompting
- Self-consistency verification
- Multi-agent verifier architectures

These approaches may improve robustness and reduce unsupported evidence attribution in multimodal reasoning systems.

# Phase 8 — Evaluation

## Question

How effectively can GeoReasoner assess business profile freshness using multimodal evidence?

## Motivation

Evaluation helps identify strengths, weaknesses, and failure modes of the system.

## Components Evaluated

- Metadata Grounding
- Rule-Based Verification
- LLM Verification
- Agentic Verification

## Metrics

- Consistency Score
- Verification Result
- Qualitative Analysis

## Evaluation Methodology

A small qualitative evaluation was performed on representative User-Generated Images.

For each image, GeoReasoner executed the complete pipeline:

Business Profile
+ Image
→ Grounding
→ Evidence Generation
→ Verification

The evaluation compares:

- Metadata claim
- Generated visual evidence
- Consistency score
- Rule-based verification
- LLM-based verification
- Agent behavior under insufficient evidence

The objective is not benchmark performance, but rather analysis of reasoning behavior and failure modes.

In [ ]:
evaluation_examples = [
    ("data/images/img_0.jpg", "restaurant"),
    ("data/images/img_1.jpg", "coffee shop"),
    ("data/images/img_2.jpg", "restaurant"),
    ("data/images/img_3.jpg", "bakery"),
    ("data/images/img_4.jpg", "grocery store"),
]

In [ ]:
for image_path, label in evaluation_examples:

    image = Image.open(image_path)

    inputs = blip_processor(
        images=image,
        return_tensors="pt"
    )

    outputs = blip_model.generate(**inputs)

    caption = blip_processor.decode(
        outputs[0],
        skip_special_tokens=True
    )

    print()
    print("Image:", image_path)
    print("Label:", label)
    print("Caption:", caption)

In [ ]:
evaluation_data = [
    {
        "image": "img_0",
        "label": "restaurant",
        "caption": "a restaurant with tables and chairs and a bar",
        "score": 0.245
    },
    {
        "image": "img_1",
        "label": "coffee shop",
        "caption": "a bicycle parked in front of a store",
        "score": 0.253
    },
    {
        "image": "img_2",
        "label": "restaurant",
        "caption": "a restaurant with a large wooden table and chairs",
        "score": 0.241
    },
    {
        "image": "img_3",
        "label": "bakery",
        "caption": "two people holding cups of coffee in their hands",
        "score": 0.204
    },
    {
        "image": "img_4",
        "label": "grocery store",
        "caption": "three plates of food on a wooden table",
        "score": 0.180
    }
]

In [ ]:
for example in evaluation_data:

  result = verify_claim_explainable(
    example["caption"],
    example["label"],
    example["score"]
  )
  print(example["image"])
  print(result)

## Evaluation Results

| Image | Business Category | Visual Evidence | Expected Verdict | Rule-Based Verifier |
|---------|---------|---------|---------|---------|
| img_0 | restaurant | restaurant with tables and chairs and a bar | SUPPORTED | INSUFFICIENT_EVIDENCE |
| img_1 | coffee shop | bicycle parked in front of a store | INSUFFICIENT_EVIDENCE | INSUFFICIENT_EVIDENCE |
| img_2 | restaurant | restaurant with a large wooden table and chairs | SUPPORTED | INSUFFICIENT_EVIDENCE |
| img_3 | bakery | two people holding cups of coffee in their hands | SUPPORTED (weak evidence) | CONTRADICTED |
| img_4 | grocery store | three plates of food on a wooden table | CONTRADICTED | CONTRADICTED |

## Evaluation Summary

Agreement Rate

2 / 5 = 40%

### Error Analysis

The rule-based verifier was highly conservative and relied primarily on keyword matching.

Common failure modes included:

- Failure to recognize semantically related evidence
- Failure to reason about context
- Limited generalization beyond predefined keywords

Examples:

- Restaurant imagery containing tables, chairs, and bar areas was classified as INSUFFICIENT_EVIDENCE.
- Coffee-related imagery was classified as CONTRADICTED for bakery despite providing weak supporting evidence.

### Key Insight

Rule-based verification provides interpretable decisions but lacks semantic understanding.

The evaluation demonstrates the tradeoff between precision and recall:

- Conservative rules reduce false positives.
- Conservative rules increase false negatives.

These findings motivated the exploration of LLM-based verification and evidence-first reasoning architectures.

In [ ]:
for example in evaluation_data:

    result = llm_verify(
        caption=example["caption"],
        metadata_label=example["label"],
        consistency_score=example["score"]
    )

    print()
    print("=" * 80)
    print(example["image"])
    print("Label:", example["label"])
    print("Caption:", example["caption"])
    print(result)

## Evaluation Results

| Image | Business Category | Visual Evidence | Expected Verdict | Rule-Based Verifier | LLM Verifier |
|---------|---------|---------|---------|---------|---------|
| img_0 | restaurant | a restaurant with tables and chairs and a bar | SUPPORTED | INSUFFICIENT_EVIDENCE | SUPPORTED |
| img_1 | coffee shop | a bicycle parked in front of a store | INSUFFICIENT_EVIDENCE | INSUFFICIENT_EVIDENCE | SUPPORTED |
| img_2 | restaurant | a restaurant with a large wooden table and chairs | SUPPORTED | INSUFFICIENT_EVIDENCE | SUPPORTED |
| img_3 | bakery | two people holding cups of coffee in their hands | WEAK_SUPPORT | CONTRADICTED | CONTRADICTED |
| img_4 | grocery store | three plates of food on a wooden table | CONTRADICTED | CONTRADICTED | SUPPORTED |

### Verdict Definitions

SUPPORTED
: The visual evidence directly supports the business category claim.

WEAK_SUPPORT
: The visual evidence is consistent with the claim but does not provide strong or definitive support.

INSUFFICIENT_EVIDENCE
: The visual evidence does not provide enough information to support or reject the claim.

CONTRADICTED
: The visual evidence is inconsistent with the business category claim.

### Evaluation Observations

- The rule-based verifier was highly conservative and frequently returned INSUFFICIENT_EVIDENCE.
- The rule-based verifier failed to recognize semantically relevant restaurant-related evidence.
- The LLM verifier correctly identified restaurant-related imagery but was susceptible to hallucination.
- The coffee-shop example revealed claim-conditioned hallucination, where the model transformed a generic storefront into a coffee shop.
- The grocery-store example demonstrated semantic overreach, where prepared food was incorrectly interpreted as evidence for a grocery store.
- The bakery example highlighted the difficulty of reasoning with weak evidence. Coffee cups may be consistent with a bakery or café but do not provide definitive proof.

## Evaluation Conclusions

The evaluation demonstrates that business profile freshness verification requires balancing semantic reasoning and verification reliability.

Rule-based verification produced conservative and interpretable decisions but suffered from low recall due to limited semantic understanding.

LLM-based verification improved semantic reasoning and correctly recognized restaurant-related imagery, but introduced hallucination failure modes including claim-conditioned reasoning, evidence injection, and semantic overreach.

Evidence-first verification improved transparency by separating evidence extraction from claim evaluation and enabled analysis of intermediate reasoning steps.

Overall, the experiments highlight the importance of combining retrieval, grounding, evidence generation, and verification when building multimodal reasoning systems for business profile freshness verification.

# Phase 9 — Agentic Freshness Verification

## Question

Can the system actively gather additional evidence when existing evidence is insufficient?

### Why Agentic Verification?

A single image often does not provide sufficient evidence to determine whether a business profile remains valid.

Instead of immediately making a decision, GeoReasoner can actively gather additional evidence and continue reasoning until one of the following conditions is met:

- SUPPORTED → Profile remains valid.
- CONTRADICTED → Potential profile drift detected.
- Maximum evidence budget reached.

## Inputs

Current Business Profile Metadata

Candidate Images

## Outputs

Final Freshness Decision. Possible values: SUPPORTED, CONTRADICTED, REVIEW_REQUIRED.

## Architecture

Current Business Profile
+ New Image
→ Grounding
→ Evidence Generation
→ Verification

SUPPORTED
→ Profile Valid
→ Stop

CONTRADICTED
→ Potential Profile Drift
→ Flag For Review
→ Stop

INSUFFICIENT_EVIDENCE
→ Retrieve Additional Evidence
→ Re-Verify
→ Repeat Until Decision

## Why This Is Agentic

The system is no longer executing a single forward pass.

Instead, it evaluates its current evidence state and decides whether additional evidence should be collected before making a final decision.

## Agent Workflow

Current Business Profile
+ New Image
→ Grounding
→ Evidence Generation
→ Verification
→ Decision

SUPPORTED
→ Keep Existing Profile

CONTRADICTED
→ Flag Profile For Review

INSUFFICIENT_EVIDENCE
→ Retrieve Additional Evidence
→ Evidence Generation
→ Verification
→ Decision

## Agent Decision Policy

SUPPORTED
→ Existing profile remains valid.
→ Stop.

CONTRADICTED
→ Potential profile drift detected.
→ Flag for review.
→ Stop.

INSUFFICIENT_EVIDENCE
→ Retrieve additional evidence.
→ Continue reasoning.

Maximum Evidence Budget Reached
→ Escalate for manual review.
→ REVIEW_REQUIRED.

### Design Choice

The agent uses the rule-based verifier rather than the LLM verifier.

Evaluation revealed that the LLM verifier was susceptible to claim-conditioned hallucination, while the rule-based verifier provided deterministic behavior suitable for iterative decision making.

#### Simplification

For demonstration purposes, candidate evidence images were manually provided.

In a production system, additional evidence would be retrieved automatically from sources such as:

- Recent User-Generated Images
- Street-level imagery
- Historical business imagery
- Nearby business observations

In [ ]:
def agentic_freshness_verification(
    image_paths,
    metadata_label,
    max_evidence_budget=3
):

    history = []

    print("=" * 80)
    print("Current Business Profile:", metadata_label)
    print("=" * 80)

    for step, image_path in enumerate(
        image_paths[:max_evidence_budget],
        start=1
    ):

        print()
        print("-" * 80)
        print(f"Evidence {step}")
        print("-" * 80)

        image = Image.open(image_path)

        inputs = blip_processor(
            images=image,
            return_tensors="pt"
        )

        outputs = blip_model.generate(**inputs)

        caption = blip_processor.decode(
            outputs[0],
            skip_special_tokens=True
        )

        score = check_consistency(
            image_path,
            metadata_label
        )

        result = verify_claim_explainable(
            caption,
            metadata_label,
            score
        )

        history.append({
            "image": image_path,
            "caption": caption,
            "score": score,
            "verification": result
        })

        print("Image:", image_path)
        print("Caption:", caption)
        print("Consistency Score:", round(score, 3))
        print("Verification:", result["verification"])
        print("Reason:", result["reason"])

        verdict = result["verification"]

        if verdict == "SUPPORTED":

            print()
            print("Agent Decision:")
            print("Profile remains valid.")
            print("Stopping evidence collection.")

            return {
                "decision": "SUPPORTED",
                "history": history
            }

        elif verdict == "CONTRADICTED":

            print()
            print("Agent Decision:")
            print("Potential profile drift detected.")
            print("Flagging profile for review.")

            return {
                "decision": "CONTRADICTED",
                "history": history
            }

        else:

            print()
            print("Agent Decision:")
            print("Insufficient evidence.")
            print("Requesting additional evidence.")

    print()
    print("=" * 80)
    print("Agent Decision:")
    print("Evidence budget exhausted.")
    print("Escalating for manual review.")

    return {
        "decision": "REVIEW_REQUIRED",
        "history": history
    }

In [ ]:
candidate_images = [
    "data/images/img_1.jpg",
    "data/images/img_3.jpg"
]

result = agentic_freshness_verification(
    image_paths=candidate_images,
    metadata_label="coffee shop"
)

## Agent Evaluation

### Example Scenario

Current Business Profile

coffee shop

Evidence 1

Caption:
"a bicycle parked in front of a store"

Verification:
INSUFFICIENT_EVIDENCE

Agent Action:
Retrieve additional evidence.

Evidence 2

Caption:
"two people holding cups of coffee in their hands"

Verification:
SUPPORTED

Agent Action:
Stop evidence collection.

Final Decision:
SUPPORTED

### Observation

The first image did not provide sufficient evidence to determine whether the business profile remained valid.

Rather than terminating, the agent gathered additional evidence and re-evaluated the claim.

The second image contained coffee-related visual evidence, allowing the system to conclude that the existing business profile remained consistent with the available imagery.

This experiment demonstrates how agentic evidence collection can improve business profile freshness verification when individual images provide incomplete information.

## Agentic Decision Loop

Current Business Profile
+ New Visual Evidence
→ Verification

SUPPORTED
→ Keep Profile

CONTRADICTED
→ Flag For Review

INSUFFICIENT_EVIDENCE
→ Gather More Evidence
→ Re-Verify

# Limitations

This project uses a small qualitative dataset and simplified metadata labels.

The objective is to explore multimodal reasoning architectures rather than achieve production-level business classification performance.

Several challenges remain:

- Limited evaluation scale
- LLM hallucination during verification
- Simplified retrieval corpus
- Evaluation uses a small manually curated image set and should be viewed as a proof-of-concept rather than a statistically significant benchmark.
- Manual evidence collection during agentic verification

# Final Project Status

Implemented Components

✅ Retrieval

✅ Metadata Grounding

✅ Visual Evidence Generation

✅ Structured Reasoning

✅ Evidence Aggregation

✅ Confidence Calibration

✅ Rule-Based Verification

✅ LLM-Based Verification

✅ Evidence-First Verification

✅ Evaluation

✅ Agentic Freshness Verification

🔄 Reflective Verification Agent

Key Contributions

- Grounded multimodal retrieval and reasoning
- Verification-aware reasoning pipeline
- Analysis of claim-conditioned hallucination
- Evidence-first verification architecture
- Comparative evaluation of rule-based and LLM-based verification

## Final Takeaway

GeoReasoner demonstrates how retrieval, grounding, evidence generation, verification, and agentic reasoning can be combined to assess business profile freshness from multimodal evidence.

The project highlights both the capabilities and limitations of LLM-based reasoning systems, particularly the importance of grounded verification, evidence-aware decision making, and iterative evidence collection when reasoning under uncertainty.